# Adult

In [10]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
import shutil
from pathlib import Path
from collections import defaultdict

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, OrdinalEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

import torch
from torch import optim
from torch import nn

from torch.utils.tensorboard import SummaryWriter

from torchmetrics.regression import MeanSquaredError, MeanAbsoluteError
from torchmetrics.classification import Accuracy, AUROC, F1Score, Recall, Precision

from catboost import CatBoostClassifier, CatBoostRegressor, Pool

from modules import LinearRegression, LogisticRegression, TabularModel, TabularDataset

from tqdm.notebook import tqdm

In [12]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

In [13]:
DATA_PATH = Path('../data')
DATASET_NAME = "adult"

ADULT_PATH = DATA_PATH / DATASET_NAME / 'adult.data'

STRATEGY_NUM = "mean"
STRATEGY_CAT = "most_frequent"

RANDOM_STATE = 0
TEST_SIZE = 0.2

In [14]:
def seed_everything(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(RANDOM_STATE)

In [15]:
shutil.rmtree(Path(DATASET_NAME), ignore_errors=True)

In [16]:
adult_columns = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education-num",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital-gain",
    "capital-loss",
    "hours-per-week",
    "native-country",
    "class",
]

adult_df = pd.read_csv(ADULT_PATH, header=None, names=adult_columns)
adult_df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,class
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [17]:
COLUMNS_NAMES = adult_df.columns.to_list()
N_COLUMNS = len(COLUMNS_NAMES)

NUM_FEATURES = np.array([0, 2, 4, 10, 11, 12])
CAT_FEATURES = np.setdiff1d(np.arange(N_COLUMNS), NUM_FEATURES)

print("Numeric features:", *NUM_FEATURES)
print("Categorical features:", *CAT_FEATURES)

Numeric features: 0 2 4 10 11 12
Categorical features: 1 3 5 6 7 8 9 13 14


По официальной документации пропуски есть только в трёх столбцах.

In [18]:
adult_df.replace(' ?', None, inplace=True)

In [19]:
adult_df.dtypes

age                int64
workclass         object
fnlwgt             int64
education         object
education-num      int64
marital-status    object
occupation        object
relationship      object
race              object
sex               object
capital-gain       int64
capital-loss       int64
hours-per-week     int64
native-country    object
class             object
dtype: object

In [20]:
adult_df[adult_df.isna().any(axis=1)].head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,class
14,40,Private,121772,Assoc-voc,11,Married-civ-spouse,Craft-repair,Husband,Asian-Pac-Islander,Male,0,0,40,None,>50K
27,54,None,180211,Some-college,10,Married-civ-spouse,None,Husband,Asian-Pac-Islander,Male,0,0,60,South,>50K
38,31,Private,84154,Some-college,10,Married-civ-spouse,Sales,Husband,White,Male,0,0,38,None,>50K
51,18,Private,226956,HS-grad,9,Never-married,Other-service,Own-child,White,Female,0,0,30,None,<=50K
61,32,None,293936,7th-8th,4,Married-spouse-absent,None,Not-in-family,White,Male,0,0,40,None,<=50K


In [21]:
def load_adult() -> TabularDataset:
    adult_df = pd.read_csv(ADULT_PATH, header=None)
    adult_df.replace(' ?', np.nan, inplace=True)
    adult_array = adult_df.to_numpy(dtype=object)

    adult_dataset = TabularDataset(adult_array, COLUMNS_NAMES, NUM_FEATURES, CAT_FEATURES)

    return adult_dataset
    

In [22]:
dataset = load_adult()

## Regression

In [23]:
def get_regression_model(dataset: TabularDataset):
    regression_model = TabularModel()

    regression_model.fit_transformers(dataset, TEST_SIZE, RANDOM_STATE)

    for target_column in range(N_COLUMNS):
        if dataset.is_num_feature(target_column):
            model = LinearRegression(N_COLUMNS - 1).to(device)

            metrics = [
                MeanSquaredError(squared=False).to(device),
                MeanAbsoluteError().to(device)
            ]

        else:
            num_classes = len(regression_model.get_categories(target_column))
            
            model = LogisticRegression(N_COLUMNS - 1, num_classes).to(device)

            metrics = [
                Accuracy(task='multiclass', num_classes=num_classes, average='macro').to(device),
                AUROC(task='multiclass', num_classes=num_classes, average='macro').to(device),
                F1Score(task='multiclass', num_classes=num_classes, average='macro').to(device),
                Recall(task='multiclass', num_classes=num_classes, average='macro').to(device),
                Precision(task='multiclass', num_classes=num_classes, average='macro').to(device)
            ]

        regression_model.models[target_column] = model
        regression_model.metrics[target_column] = metrics
    
    return regression_model

In [24]:
regression_model = get_regression_model(dataset)

Fitting column:   0%|          | 0/15 [00:00<?, ?it/s]

Fitting column within:   0%|          | 0/15 [00:00<?, ?it/s]

Fitting column within:   0%|          | 0/15 [00:00<?, ?it/s]

Fitting column within:   0%|          | 0/15 [00:00<?, ?it/s]

Fitting column within:   0%|          | 0/15 [00:00<?, ?it/s]

...

Fitting column within:   0%|          | 0/15 [00:00<?, ?it/s]

Fitting column within:   0%|          | 0/15 [00:00<?, ?it/s]

Fitting column within:   0%|          | 0/15 [00:00<?, ?it/s]

Fitting column within:   0%|          | 0/15 [00:00<?, ?it/s]

Fitting column within:   0%|          | 0/15 [00:00<?, ?it/s]

In [25]:
def train_regression_model(regression_model: TabularModel,
                            dataset: TabularDataset,
                            epochs: int = 10000):
    
    for target_column in tqdm(range(N_COLUMNS), desc="Column", leave=False):
        X_y_train, X_y_valid = dataset.get_train_test_for_column(target_column, TEST_SIZE, RANDOM_STATE)
        X_y_train = regression_model.transform(X_y_train, target_column, with_target_column=True)
        X_y_valid = regression_model.transform(X_y_valid, target_column, with_target_column=True)

        X_train = X_y_train[:, regression_model.get_mask(target_column)]
        y_train = X_y_train[:, target_column]

        X_valid = X_y_valid[:, regression_model.get_mask(target_column)]
        y_valid = X_y_valid[:, target_column]

        X_train = torch.from_numpy(X_train.astype(float)).to(device, torch.float32)
        X_valid = torch.from_numpy(X_valid.astype(float)).to(device, torch.float32)

        if regression_model.is_num_feature(target_column):
            y_train = torch.from_numpy(y_train.astype(float)).to(device, torch.float32)
            y_valid = torch.from_numpy(y_valid.astype(float)).to(device, torch.float32)
            criterion = nn.MSELoss()
        else:
            y_train = torch.from_numpy(y_train.astype(int)).to(torch.long)
            y_valid = torch.from_numpy(y_valid.astype(int)).to(torch.long)
            criterion = nn.CrossEntropyLoss()
        
        # ------------------- TRAINING ----------------------
        model = regression_model.models[target_column]
        optimizer = optim.Adam(model.parameters(), lr=1e-3)

        best_val_loss = float("inf")
        best_state_dict = None
        best_epoch = -1
        
        for epoch in tqdm(range(1, epochs + 1), "Training", leave=False):
            # ---------------------- TRAIN ------------------------
            model.train()

            optimizer.zero_grad()
            prediction = model(X_train)
            loss = criterion(prediction, y_train)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            # ---------------------- VALID -------------------------
            model.eval()
            with torch.no_grad():
                pred_valid = model(X_valid)
                val_loss = criterion(pred_valid, y_valid).item()

                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    best_epoch = epoch
                    best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if best_state_dict is not None:
            model.load_state_dict(best_state_dict)
        print(f"Target column = {target_column}, "
              f"Type = {model.__class__.__name__ }, "
              f"Best epoch = {best_epoch}, "
              f"Best val loss={best_val_loss:.6f}")

In [26]:
train_regression_model(regression_model, dataset, epochs=1000)

Column:   0%|          | 0/15 [00:00<?, ?it/s]

Training:   0%|          | 0/1000 [00:00<?, ?it/s]

Target column = 0, Type = LinearRegression, Best epoch = 1000, Best val loss=0.845988


Training:   0%|          | 0/1000 [00:00<?, ?it/s]

Target column = 1, Type = LogisticRegression, Best epoch = 1000, Best val loss=0.912246


...

Target column = 12, Type = LinearRegression, Best epoch = 361, Best val loss=0.943034


Training:   0%|          | 0/1000 [00:00<?, ?it/s]

Target column = 13, Type = LogisticRegression, Best epoch = 1000, Best val loss=0.499203


Training:   0%|          | 0/1000 [00:00<?, ?it/s]

Target column = 14, Type = LogisticRegression, Best epoch = 1000, Best val loss=0.392000


In [27]:
def bootstrap_regression(regression_model: TabularModel,
                         dataset: TabularDataset,
                         iterations: int = 1000):
    
    output = list()
    
    for target_column in tqdm(range(N_COLUMNS), desc="Column", leave=False):

        model = regression_model.models[target_column]
        metrics = regression_model.metrics[target_column]
        metrics_by_name = defaultdict(list)

        _, X_y_test = dataset.get_train_test_for_column(target_column, test_size=0.2, random_state=RANDOM_STATE)
        X_y_test = regression_model.transform(X_y_test, target_column, with_target_column=True)
        
        for metric in metrics:
            name = metric.__class__.__name__

            for _ in tqdm(range(iterations), desc="Bootstrap", leave=False):
                bootstrap = np.random.choice(X_y_test.shape[0], X_y_test.shape[0], replace=True)
                X_test = X_y_test[:, regression_model.get_mask(target_column)]
                X_test = X_test[bootstrap, :]
                y_test = X_y_test[bootstrap, target_column]
                y_test = y_test[bootstrap]

                X_test = torch.from_numpy(X_test.astype(float)).to(device, torch.float32)
                if regression_model.is_num_feature(target_column):
                    y_pred = model.predict(X_test)
                    y_test = torch.from_numpy(y_test.astype(float)).to(device, torch.float32)
                else:
                    y_pred = model.predict_proba(X_test)
                    y_test = torch.from_numpy(y_test.astype(int)).to(device, torch.long)

                metric.reset()
                value = metric(y_pred, y_test)
                metrics_by_name[name].append(value.item())
        
        for name, values in metrics_by_name.items():
            values = np.array(values, dtype=float)

            row = {
                "column": dataset.columns_names[target_column],
                "model": "regression",
                "metric": name,
                "mean": values.mean(),
                "std": values.std(ddof=1) if values.size > 1 else 0.0
            }

            output.append(row)

    output = pd.DataFrame(output)
    return output

In [28]:
regression_model_statistics = bootstrap_regression(regression_model, dataset, iterations=100)

Column:   0%|          | 0/15 [00:00<?, ?it/s]

Bootstrap:   0%|          | 0/100 [00:00<?, ?it/s]

Bootstrap:   0%|          | 0/100 [00:00<?, ?it/s]

Bootstrap:   0%|          | 0/100 [00:00<?, ?it/s]

Bootstrap:   0%|          | 0/100 [00:00<?, ?it/s]

...

Bootstrap:   0%|          | 0/100 [00:00<?, ?it/s]

Bootstrap:   0%|          | 0/100 [00:00<?, ?it/s]

Bootstrap:   0%|          | 0/100 [00:00<?, ?it/s]

Bootstrap:   0%|          | 0/100 [00:00<?, ?it/s]

Bootstrap:   0%|          | 0/100 [00:00<?, ?it/s]

## CatBoost

In [42]:
def get_catboost(dataset: TabularDataset, iterations: int = 100, learning_rate: float = 1e-3):
    catboost_model = TabularModel()

    catboost_model.fit_transformers(dataset, TEST_SIZE, RANDOM_STATE)

    task_type = "GPU" if torch.cuda.is_available() else "CPU"

    for target_column in range(N_COLUMNS):
        if dataset.is_num_feature(target_column):
            model = CatBoostRegressor(iterations=iterations,
                                      learning_rate=learning_rate,
                                      task_type=task_type,
                                      allow_writing_files=False)
            
            metrics = [
                MeanSquaredError(squared=False).to(device),
                MeanAbsoluteError().to(device)
            ]

        else:
            num_classes = len(catboost_model.get_categories(target_column))

            model = CatBoostClassifier(iterations=iterations,
                                       learning_rate=learning_rate,
                                       loss_function='MultiClass',
                                       classes_count=num_classes,
                                       task_type=task_type,
                                       allow_writing_files=False)
            
            metrics = [
                Accuracy(task='multiclass', num_classes=num_classes, average='macro').to(device),
                AUROC(task='multiclass', num_classes=num_classes, average='macro').to(device),
                F1Score(task='multiclass', num_classes=num_classes, average='macro').to(device),
                Recall(task='multiclass', num_classes=num_classes, average='macro').to(device),
                Precision(task='multiclass', num_classes=num_classes, average='macro').to(device)
            ]

        catboost_model.models[target_column] = model
        catboost_model.metrics[target_column] = metrics
    
    return catboost_model

In [47]:
catboost_model = get_catboost(dataset, iterations=10)

Fitting column:   0%|          | 0/15 [00:00<?, ?it/s]

Fitting column within:   0%|          | 0/15 [00:00<?, ?it/s]

Fitting column within:   0%|          | 0/15 [00:00<?, ?it/s]

Fitting column within:   0%|          | 0/15 [00:00<?, ?it/s]

Fitting column within:   0%|          | 0/15 [00:00<?, ?it/s]

...

Fitting column within:   0%|          | 0/15 [00:00<?, ?it/s]

Fitting column within:   0%|          | 0/15 [00:00<?, ?it/s]

Fitting column within:   0%|          | 0/15 [00:00<?, ?it/s]

Fitting column within:   0%|          | 0/15 [00:00<?, ?it/s]

Fitting column within:   0%|          | 0/15 [00:00<?, ?it/s]

In [ ]:
def train_catboost_model(catboost_model: TabularModel,
                         dataset: TabularDataset):
    
    for target_column in tqdm(range(N_COLUMNS), desc="Column", leave=False):
        X_y_train, X_y_valid = dataset.get_train_test_for_column(target_column, TEST_SIZE, RANDOM_STATE)

        X_y_train = catboost_model.transform(X_y_train, target_column, impute_num_features=False)
        X_y_valid = catboost_model.transform(X_y_valid, target_column, impute_num_features=False)

        X_train = X_y_train[:, catboost_model.get_mask(target_column)]
        y_train = X_y_train[:, target_column]
        X_valid = X_y_valid[:, catboost_model.get_mask(target_column)]
        y_valid = X_y_valid[:, target_column]

        cat_features = catboost_model.get_shifted_cat_features(target_column)

        train_pool = Pool(X_train, y_train, cat_features)
        valid_pool = Pool(X_valid, y_valid, cat_features)
        
        catboost_model.models[target_column].fit(train_pool, eval_set=valid_pool, use_best_model=True, verbose=False)

In [48]:
train_catboost_model(catboost_model, dataset)

Column:   0%|          | 0/15 [00:00<?, ?it/s]

In [ ]:
def bootstrap_catboost(catboost_model: TabularModel,
                       dataset: TabularDataset,
                       iterations: int = 1000):
    
    output = list()
    
    for target_column in tqdm(range(N_COLUMNS), desc="Column", leave=False):

        model = catboost_model.models[target_column]
        metrics = catboost_model.metrics[target_column]
        metrics_by_name = defaultdict(list)

        _, X_y_test = dataset.get_train_test_for_column(target_column, TEST_SIZE, RANDOM_STATE)
        X_y_test = catboost_model.transform(X_y_test, target_column, impute_num_features=False)

        for metric in metrics:
            name = metric.__class__.__name__

            for _ in tqdm(range(iterations), desc="Bootstrap", leave=False):
                bootstrap = np.random.choice(X_y_test.shape[0], X_y_test.shape[0], replace=True)
                X_test = X_y_test[:, catboost_model.get_mask(target_column)]
                X_test = X_test[bootstrap, :]
                y_test = X_y_test[bootstrap, target_column]
                y_test = y_test[bootstrap]

                if catboost_model.is_num_feature(target_column):
                    y_pred = model.predict(X_test)
                    y_pred = torch.from_numpy(y_pred.astype(float)).to(device, dtype=torch.float32)
                    y_test = torch.from_numpy(y_test.astype(float)).to(device, dtype=torch.float32)
                else:
                    y_pred = model.predict_proba(X_test)
                    y_pred = torch.from_numpy(y_pred.astype(float)).to(device, dtype=torch.float32)
                    y_test = torch.from_numpy(y_test.astype(int)).to(device, dtype=torch.long)

                metric.reset()
                value = metric(y_pred, y_test)
                metrics_by_name[name].append(value.item())

        for name, values in metrics_by_name.items():
            values = np.array(values, dtype=float)

            row = {
                "column": dataset.columns_names[target_column],
                "model": "catboost",
                "metric": name,
                "mean": values.mean(),
                "std": values.std(ddof=1) if values.size > 1 else 0.0
            }

            output.append(row)
    
    output = pd.DataFrame(output)
    return output

In [ ]:
catboost_model_statistics = bootstrap_catboost(catboost_model, dataset, iterations=1000)

# Compare

In [ ]:
def get_compare_table(regression_model_statistics: pd.DataFrame, catboost_model_statistics: pd.DataFrame):
    out = pd.concat([regression_model_statistics, catboost_model_statistics], axis=0)
    out = out.set_index(["column", "metric", "model"]).sort_index()
    out = out.round(5)
    return out

In [ ]:
catboost_vs_regression = get_compare_table(regression_model_statistics, catboost_model_statistics)
catboost_vs_regression.to_csv(f"{DATASET_NAME}_regression_vs_catboost", index=True)

In [ ]:
catboost_vs_regression.head(15)